# Real-time GSN-ISEM demo

The Teukolsky wrapper stores the underlying Direct GSN function, so the demo constructs one route and evaluates both \(R(r)\) and \(X(r_*)\).


In [ ]:
using GeneralizedSasakiNakamura
using Plots, LaTeXStrings, Printf


In [ ]:
function radial_demo_frame(; boundary = IN, s = -2, l = 2, m = 2, a = 0.7, omega = 0.5)
    solve_time = @elapsed begin
        R = Teukolsky_radial(s, l, m, a, omega, boundary; method = "GSN-ISEM")
        X = R.GSN_solution
    end

    rstar_grid = collect(range(-20, 100; length = 1600))
    r_grid = r_from_rstar.(a, rstar_grid)

    eval_time = @elapsed begin
        R_values = R.(r_grid)
        X_values = X.(rstar_grid)
    end

    pR = plot(r_grid, [real.(R_values) imag.(R_values)];
        xlabel = L"r/M", ylabel = L"R(r)",
        title = "Teukolsky $boundary solution", label = ["real" "imag"],
        legend = :outerbottomright, legendcolumns = 2)

    pX = plot(rstar_grid, [real.(X_values) imag.(X_values)];
        xlabel = L"r_*/M", ylabel = L"X(r_*)",
        title = "Direct GSN route", label = ["real" "imag"],
        legend = :outerbottomright, legendcolumns = 2)

    summary = plot(; border = :none, ticks = nothing, annotation = [
        (0.18, 0.76, (@sprintf("incidence: %.4e %+.4e i", real(R.incidence_amplitude), imag(R.incidence_amplitude)), 8)),
        (0.50, 0.76, (@sprintf("reflection: %.4e %+.4e i", real(R.reflection_amplitude), imag(R.reflection_amplitude)), 8)),
        (0.82, 0.76, (@sprintf("transmission: %.4e %+.4e i", real(R.transmission_amplitude), imag(R.transmission_amplitude)), 8)),
        (0.50, 0.28, (@sprintf("construction: %.3f ms    evaluation on %d points: %.3f ms",
            1000 * solve_time, length(r_grid), 1000 * eval_time), 8)),
    ])

    plot(pR, pX, summary; layout = (3, 1), size = (900, 760))
end


In [ ]:
radial_demo_frame()
